# Pulmonary fibrosis simulation — interactive runner

Runs the coupled alveolar epithelium, collapse, breathing, matrix, and
confined-mesenchyme model from `Danpc11/lung-nematic`.

Run the cells in order. Results are cleared at the start of a new run by
default, the exact Git commit is recorded, and every exported bundle
contains the configuration and diagnostics needed to audit that run.

This is a mechanistic simulation, not a calibrated patient predictor.
Read `simulations/alveolar/README.md` before interpreting its outputs.

In [ ]:
#@title 1 · Setup repository and dependencies { display-mode: "form" }
import os
import subprocess
import sys

REPO = "https://github.com/Danpc11/lung-nematic.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
ROOT = "/content/lung-nematic"

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(
        ["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", ROOT, "reset", "--hard", f"origin/{BRANCH}"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO, ROOT],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ROOT,
     "imageio", "imageio-ffmpeg", "seaborn"],
    check=True,
)

for name in [module for module in list(sys.modules)
             if module == "simulations" or module.startswith("simulations.")]:
    del sys.modules[name]
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulations.alveolar import AlveolarConfig, run_and_record_coupled

commit = subprocess.run(
    ["git", "-C", ROOT, "rev-parse", "HEAD"], check=True,
    capture_output=True, text=True,
).stdout.strip()
print(f"Repository ready at commit {commit[:12]}")
print(f"{len(AlveolarConfig.__dataclass_fields__)} parameters available")

## Parameters

The controls are divided into scenario, biology, and output. The director
coarse-graining radius is exposed explicitly because defect counts below
the counting-noise limit must not be interpreted.

In [ ]:
#@title 2 · Scenario and simulated time { display-mode: "form" }
years_to_simulate = 2.0  #@param {type:"slider", min:0.25, max:5, step:0.25}
injury_duration_months = 3.3  #@param {type:"slider", min:0, max:24, step:0.1}
timestep_hours = 2.0  #@param {type:"number"}

repair_failure_factor = 0.10  #@param {type:"slider", min:0.02, max:1.0, step:0.01}
injury_activation_boost = 10.0  #@param {type:"slider", min:1, max:30, step:1}
injury_radius_um = 260.0  #@param {type:"slider", min:80, max:500, step:10}
rate_scale = 0.08  #@param {type:"slider", min:0.01, max:1.0, step:0.01}
domain_um = 1400.0  #@param {type:"slider", min:800, max:2200, step:100}
random_seed = 0  #@param {type:"integer"}

scenario = dict(
    total_time_h=float(years_to_simulate) * 8760.0,
    injury_duration_h=float(injury_duration_months) * 730.0,
    dt_h=float(timestep_hours),
    repair_failure_factor=float(repair_failure_factor),
    injury_activation_boost=float(injury_activation_boost),
    injury_radius_um=float(injury_radius_um),
    rate_scale=float(rate_scale),
    width_um=float(domain_um),
    height_um=float(domain_um),
    seed=int(random_seed),
)
print(
    f"{years_to_simulate:.2f} years; differentiation block lifted at "
    f"month {injury_duration_months:.1f}"
)

In [ ]:
#@title 3 · Biology, mechanics, and director resolution { display-mode: "form" }
stall_promotion_strength = 25.0  #@param {type:"slider", min:0, max:40, step:0.5}
repair_inhibition_strength = 3.0  #@param {type:"slider", min:0, max:10, step:0.5}
aberrant_clearance_rate = 0.0005  #@param {type:"number"}
aberrant_emt_rate = 0.004  #@param {type:"number"}

deposition_rate_kPa_per_h = 0.16  #@param {type:"number"}
degradation_rate_per_h = 0.004  #@param {type:"number"}
E_act_kPa = 16.0  #@param {type:"slider", min:4, max:40, step:0.5}

tidal_strain = 0.10  #@param {type:"slider", min:0.02, max:0.25, step:0.01}
strain_protection_strength = 6.0  #@param {type:"slider", min:0, max:20, step:0.5}
strain_tgfb_gain = 0.8  #@param {type:"slider", min:0, max:3, step:0.1}
overstrain_threshold = 0.16  #@param {type:"slider", min:0.05, max:0.4, step:0.01}

cell_length_um = 50.0  #@param {type:"slider", min:20, max:90, step:5}
cell_width_um = 11.0  #@param {type:"slider", min:5, max:25, step:1}
coarse_grain_um = 28.0  #@param {type:"slider", min:15, max:150, step:1}
fibroblast_death_rate = 0.002  #@param {type:"number"}
myofibroblast_death_rate = 0.0004  #@param {type:"number"}
matrix_immobilization = 25.0  #@param {type:"slider", min:0, max:200, step:5}

tissue_recoil_Pa = 420.0  #@param {type:"slider", min:200, max:800, step:10}
induration_time_h = 240.0  #@param {type:"number"}

biology = dict(
    stall_promotion_strength=float(stall_promotion_strength),
    repair_inhibition_strength=float(repair_inhibition_strength),
    aberrant_clearance_rate=float(aberrant_clearance_rate),
    aberrant_emt_rate=float(aberrant_emt_rate),
    deposition_rate_kPa_per_h=float(deposition_rate_kPa_per_h),
    degradation_rate_per_h=float(degradation_rate_per_h),
    E_act_kPa=float(E_act_kPa),
    tidal_strain=float(tidal_strain),
    strain_protection_strength=float(strain_protection_strength),
    strain_tgfb_gain=float(strain_tgfb_gain),
    overstrain_threshold=float(overstrain_threshold),
    cell_length_um=float(cell_length_um),
    cell_width_um=float(cell_width_um),
    coarse_grain_um=float(coarse_grain_um),
    fibroblast_death_rate=float(fibroblast_death_rate),
    myofibroblast_death_rate=float(myofibroblast_death_rate),
    matrix_immobilization=float(matrix_immobilization),
    tissue_recoil_Pa=float(tissue_recoil_Pa),
    induration_time_h=float(induration_time_h),
)

cell_area = np.pi * 0.25 * cell_length_um * cell_width_um
minimum_reliable_radius = np.sqrt(30 * cell_area / np.pi)
print(
    f"Cell footprint {cell_area:.0f} µm²; approximately 30 cells require "
    f"a radius ≥ {minimum_reliable_radius:.0f} µm."
)
if coarse_grain_um < minimum_reliable_radius:
    print(
        "WARNING: coarse_grain_um is below the counting-noise estimate. "
        "Defect markers may be visual artefacts and must be checked in diagnostics."
    )
else:
    print("Director window is at or above the geometric counting-noise estimate.")

In [ ]:
#@title 4 · Visualisation and output { display-mode: "form" }
frames_per_run = 60  #@param {type:"slider", min:10, max:150, step:5}
frames_per_second = 8  #@param {type:"slider", min:2, max:24, step:1}
resolution_dpi = 110  #@param {type:"slider", min:60, max:200, step:10}
show_strain_panel = True  #@param {type:"boolean"}
show_defects = True  #@param {type:"boolean"}
cell_transparency = 0.55  #@param {type:"slider", min:0.1, max:1.0, step:0.05}
stiffness_colormap = "pink_r"  #@param ["pink_r", "inferno", "magma", "YlOrBr", "bone_r"]
strain_colormap = "RdBu_r"  #@param ["RdBu_r", "coolwarm", "seismic", "PuOr"]
export_mp4 = True  #@param {type:"boolean"}
export_gif = False  #@param {type:"boolean"}
clear_previous_results = True  #@param {type:"boolean"}

visualisation = dict(
    fps=int(frames_per_second),
    dpi=int(resolution_dpi),
    show_strain_panel=bool(show_strain_panel),
    show_defects=bool(show_defects),
    cell_alpha=float(cell_transparency),
    stiffness_cmap=stiffness_colormap,
    strain_cmap=strain_colormap,
    make_mp4=bool(export_mp4),
    make_gif=bool(export_gif),
)
print(
    f"Requested {frames_per_run} intervals at {frames_per_second} fps; "
    f"video duration ≈ {(frames_per_run + 1) / frames_per_second:.1f} s."
)

In [ ]:
#@title 5 · Run the simulation
import shutil
import time
from dataclasses import replace
from pathlib import Path

RESULTS_DIR = Path("/content/results")
if clear_previous_results and RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

config = replace(AlveolarConfig(), **scenario, **biology)
config.validate()
n_steps = int(round(config.total_time_h / config.dt_h))
frame_every_h = config.total_time_h / int(frames_per_run)
steps_per_frame = max(1, int(round(frame_every_h / config.dt_h)))
expected_frames = 1 + n_steps // steps_per_frame
estimated_seconds = n_steps * 0.011 + expected_frames * (
    2.3 if show_strain_panel else 1.2
)
print(
    f"{n_steps:,} steps, approximately {expected_frames} frames; "
    f"estimated runtime {estimated_seconds / 60:.1f} min. Starting...\n"
)

started = time.time()

def show_progress(number):
    fraction = min(number / expected_frames, 1.0)
    bar = "#" * int(fraction * 30)
    print(
        f"\r  [{bar:<30}] {fraction * 100:5.1f}%  "
        f"{time.time() - started:5.0f}s",
        end="", flush=True,
    )

outputs = run_and_record_coupled(
    config,
    RESULTS_DIR,
    frame_every_h=frame_every_h,
    progress=show_progress,
    **visualisation,
)
elapsed_seconds = time.time() - started
print(f"\n\nFinished in {elapsed_seconds:.0f} s; {outputs['n_frames']} frames written.")

frame = pd.read_csv(outputs["timeseries"])
final = outputs["final"]
print(f"\nAt {final['time_d'] / 365:.2f} years:")
print(f"  AT1 coverage         {final['frac_AT1'] * 100:5.1f} %")
print(f"  aberrant basaloid    {final['frac_aberrant'] * 100:5.1f} %")
print(f"  alveoli indurated    {final['frac_indurated'] * 100:5.1f} %")
print(f"  mesenchymal cells    {int(final['n_mesenchymal']):5d} "
      f"({int(final['n_myofibroblast'])} myofibroblast)")
print(f"  peak stiffness       {final['max_stiffness_kPa']:5.1f} kPa")
print(f"  septal thickness     {final['mean_septal_thickness_um']:5.1f} µm")
print(f"  strain amplification {final['strain_amplification']:5.2f} ×")

RUN_INFO = {
    "git_commit": commit,
    "elapsed_seconds": elapsed_seconds,
    "expected_frames": expected_frames,
    "actual_frames": int(outputs["n_frames"]),
}

In [ ]:
#@title 6 · Preview the simulation
from IPython.display import Image, Video, display

if "mp4" in outputs:
    display(Video(outputs["mp4"], embed=True, width=960, html_attributes="controls loop"))
elif "gif" in outputs:
    display(Image(filename=outputs["gif"]))
else:
    print("No video was requested. Frames remain under /content/results/frames.")

In [ ]:
#@title 7 · Publication-style time courses { display-mode: "form" }
figure_width = "double column (183 mm)"  #@param ["single column (89 mm)", "double column (183 mm)"]
save_vector_pdf = True  #@param {type:"boolean"}

import matplotlib as mpl
import seaborn as sns

palette = {
    "blue": "#0072B2", "vermillion": "#D55E00", "green": "#009E73",
    "orange": "#E69F00", "sky": "#56B4E9", "pink": "#CC79A7",
    "brown": "#8C6D5B", "grey": "#999999",
}
width_mm = 89.0 if figure_width.startswith("single") else 183.0
width_in = width_mm / 25.4
sns.set_theme(style="ticks", context="paper")
mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans", "Arial"],
    "font.size": 7, "axes.labelsize": 7, "axes.titlesize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "legend.frameon": False, "axes.linewidth": 0.5,
    "lines.linewidth": 1.1, "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02, "pdf.fonttype": 42, "ps.fonttype": 42,
})

months = frame["time_d"] / 30.4
block_lifted = config.injury_duration_h / 730.0
single = figure_width.startswith("single")
figure, axes = plt.subplots(
    2, 2, figsize=(width_in, width_in * (0.95 if single else 0.55)), sharex=True
)

def finish(axis, ylabel, letter, legend=True):
    sns.despine(ax=axis, trim=False)
    axis.set_ylabel(ylabel)
    axis.axvline(block_lifted, color="0.6", lw=0.5, ls=(0, (2, 2)))
    axis.text(-0.20, 1.04, letter, transform=axis.transAxes,
              fontsize=8, fontweight="bold", va="bottom", ha="left")
    if legend:
        axis.legend(loc="best", handlelength=1.2, borderpad=0.2)

axis = axes[0, 0]
axis.plot(months, frame["frac_AT1"] * 100, color=palette["green"], label="AT1")
axis.plot(months, frame["frac_aberrant"] * 100, color=palette["vermillion"], label="aberrant")
axis.plot(months, frame["frac_KRT8"] * 100, color=palette["orange"], label="KRT8+")
axis.plot(months, frame["frac_denuded"] * 100, color=palette["grey"], label="denuded")
finish(axis, "epithelium (%)", "a")

axis = axes[0, 1]
axis.plot(months, frame["frac_open"] * 100, color=palette["blue"], label="open")
axis.plot(months, frame["frac_collapsed"] * 100, color=palette["sky"], label="collapsed")
axis.plot(months, frame["frac_indurated"] * 100, color=palette["brown"], label="indurated")
finish(axis, "alveoli (%)", "b")

axis = axes[1, 0]
axis.plot(months, frame["n_mesenchymal"], color=palette["blue"], label="mesenchymal")
axis.plot(months, frame["n_myofibroblast"], color=palette["vermillion"], label="myofibroblast")
finish(axis, "cells (n)", "c")
axis.set_xlabel("time (months)")

axis = axes[1, 1]
axis.plot(months, frame["max_stiffness_kPa"], color=palette["brown"], label="peak stiffness")
axis.plot(months, frame["mean_septal_thickness_um"], color=palette["pink"], label="septal thickness")
twin = axis.twinx()
twin.plot(months, frame["strain_amplification"], color=palette["sky"],
          ls=(0, (3, 1.5)), label="strain amplification")
twin.set_ylabel("strain / normal")
twin.spines["top"].set_visible(False)
finish(axis, "kPa | µm", "d", legend=False)
axis.set_xlabel("time (months)")
handles = axis.get_lines() + twin.get_lines()
axis.legend(handles, [item.get_label() for item in handles], loc="upper left")

figure.align_ylabels(axes)
figure.tight_layout(pad=0.4, w_pad=1.6, h_pad=0.8)
figure.savefig(RESULTS_DIR / "time_courses.png", dpi=600)
if save_vector_pdf:
    figure.savefig(RESULTS_DIR / "time_courses.pdf")
plt.show()
print(f"Figure saved at {width_mm:.0f} mm wide.")

In [ ]:
#@title 8 · Diagnostics: counting noise and focus localisation { display-mode: "form" }
import json
from scipy.spatial import cKDTree
from simulations.alveolar.model import OPEN

simulation = outputs["simulation"]
mesenchyme = simulation.mesenchyme
epithelium = simulation.epithelium
cell_area = config.cell_area_um2
minimum_radius = np.sqrt(30 * cell_area / np.pi)

if mesenchyme.n_cells:
    points = np.column_stack([mesenchyme.x, mesenchyme.y])
    tree = cKDTree(points)
    neighbours = np.array([
        len(tree.query_ball_point(point, config.coarse_grain_um))
        for point in points
    ])
    director = mesenchyme.director_field()
    packing = director["density"] * cell_area / mesenchyme.grid_step ** 2
    gated = packing >= config.min_packing_for_nematic
    measured_order = (
        float(np.median(director["order"][gated])) if gated.any() else float("nan")
    )
    cells_per_window = float(np.median(neighbours))
else:
    measured_order = float("nan")
    cells_per_window = 0.0
noise_floor = 1.0 / np.sqrt(max(cells_per_window, 1.0))
above_noise_floor = bool(np.isfinite(measured_order) and measured_order > noise_floor)

print("COUNTING-NOISE FLOOR")
print(f"  required geometric radius  {minimum_radius:7.1f} µm")
print(f"  configured radius          {config.coarse_grain_um:7.1f} µm")
print(f"  cells per window (median)  {cells_per_window:7.1f}")
print(f"  random-order floor         {noise_floor:7.3f}")
print(f"  measured median order      {measured_order:7.3f}")
print("  interpretation:", "above floor" if above_noise_floor else "NOISE-LIMITED")

labels = mesenchyme.alveolus_label
if mesenchyme.n_cells:
    ix = np.clip((mesenchyme.x / mesenchyme.grid_step).astype(int), 0, mesenchyme.nx - 1)
    iy = np.clip((mesenchyme.y / mesenchyme.grid_step).astype(int), 0, mesenchyme.ny - 1)
    owners = labels[iy, ix]
    inside = ((owners >= 0) &
              (epithelium.alveolar_state[np.clip(owners, 0, None)] != OPEN))
    fraction_inside = float(inside.mean())
    inside_count = int(inside.sum())
else:
    fraction_inside = 0.0
    inside_count = 0
valid = labels >= 0
collapsed_area = float(
    (epithelium.alveolar_state[labels[valid]] != OPEN).mean()
) if valid.any() else 0.0
enrichment = (
    fraction_inside / collapsed_area if collapsed_area > 0 else float("nan")
)

print("\nFOCUS LOCALISATION")
print(f"  cells inside collapsed alveoli  {inside_count}/{mesenchyme.n_cells}")
print(f"  fraction inside                 {fraction_inside:.3f}")
print(f"  collapsed share of area         {collapsed_area:.3f}")
print(f"  enrichment over chance          {enrichment:.3f}")
print("\nOne seed is one realisation; repeat seeds before reporting an effect size.")

def finite_or_none(value):
    return float(value) if np.isfinite(value) else None

DIAGNOSTICS = {
    "minimum_reliable_radius_um": float(minimum_radius),
    "configured_coarse_grain_um": float(config.coarse_grain_um),
    "cells_per_window_median": cells_per_window,
    "noise_floor": float(noise_floor),
    "measured_order": finite_or_none(measured_order),
    "above_noise_floor": above_noise_floor,
    "mesenchymal_fraction_inside_collapsed": fraction_inside,
    "collapsed_area_fraction": collapsed_area,
    "focus_enrichment": finite_or_none(enrichment),
}
with (RESULTS_DIR / "diagnostics.json").open("w", encoding="utf-8") as handle:
    json.dump(DIAGNOSTICS, handle, indent=2, allow_nan=False)

In [ ]:
#@title 9 · Retrospective drug controls { display-mode: "form" }
dosing = "treatment (established disease)"  #@param ["prevention (from day 0)", "treatment (established disease)"]
treatment_start_months = 18.0  #@param {type:"slider", min:1, max:35, step:1}
control_duration_months = 36.0  #@param {type:"slider", min:12, max:60, step:1}

from simulations.coupled_analysis import CoupledParameters
from simulations.pharmacology import run_panel_with_maturation, score

reduced = CoupledParameters.from_alveolar(config)
treatment_start_h = (
    0.0 if dosing.startswith("prevention")
    else float(treatment_start_months) * 730.0
)
control_total_h = float(control_duration_months) * 730.0
if treatment_start_h >= control_total_h:
    raise ValueError("Treatment must start before the control protocol ends.")

# Delayed dosing requires the maturation model: a legacy lumped matrix
# cannot distinguish preventing new crosslinks from reversing mature scar.
panel = run_panel_with_maturation(
    reduced,
    treatment_start_h=treatment_start_h,
    total_time_h=control_total_h,
    dt_h=config.dt_h,
)
print(
    f"Reduced-model protocol: {control_duration_months:.0f} months; "
    f"drug start at {treatment_start_h / 730.0:.0f} months."
)
display(panel)
control_score = score(panel)
print("Clinical failures reproduced:",
      f"{control_score['clinical_failures_reproduced']}/"
      f"{control_score['clinical_failures_total']}")
print("Passes discriminating test:", control_score["passes_discriminating_test"])
panel.to_csv(RESULTS_DIR / "drug_controls.csv", index=False)
with (RESULTS_DIR / "drug_control_score.json").open("w", encoding="utf-8") as handle:
    json.dump(control_score, handle, indent=2)

In [ ]:
#@title 10 · Export the current run { display-mode: "form" }
import json
import zipfile
from dataclasses import asdict

bundle_name = "ipf_simulation_run"  #@param {type:"string"}
also_include_frames = False  #@param {type:"boolean"}

safe_bundle_name = Path(bundle_name).name.strip() or "ipf_simulation_run"
bundle = Path("/content") / f"{safe_bundle_name}.zip"
config.to_json(RESULTS_DIR / "config.json")
frame.to_csv(RESULTS_DIR / "timeseries.csv", index=False)
manifest = {
    **RUN_INFO,
    "config": asdict(config),
    "media": {"mp4": "mp4" in outputs, "gif": "gif" in outputs},
    "frames_in_bundle": bool(also_include_frames),
}
with (RESULTS_DIR / "run_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for item in sorted(RESULTS_DIR.rglob("*")):
        if not item.is_file():
            continue
        relative = item.relative_to(RESULTS_DIR)
        if relative.parts[0] == "frames" and not also_include_frames:
            continue
        archive.write(item, str(relative))

print(f"{bundle.name} ({bundle.stat().st_size / 1e6:.1f} MB)")
print("Contains only the current run: config, manifest, time series, diagnostics, figures, and requested media.")
from google.colab import files

files.download(str(bundle))

## Reproducing and interpreting a run

`config.json` contains the simulation parameters and `run_manifest.json`
records the exact Git commit. Recreate a run with:

```python
import json
from dataclasses import replace
from simulations.alveolar import AlveolarConfig, run_and_record_coupled

with open("config.json") as handle:
    config = replace(AlveolarConfig(), **json.load(handle))
config.validate()
run_and_record_coupled(config, "results/rerun", frame_every_h=292.0)
```

Treat defect counts as measurements only when cell 8 reports that order is
above the counting-noise floor. Focus localisation, collapse, architecture,
septal thickening, and strain redistribution are separate measurements.
Drug controls use the reduced maturation model and a protocol stated in
months; they do not reuse the agent simulation's spatial state.

Every output is one random seed. Repeat across seeds before quoting an
effect size or comparing interventions.